In [1]:
import os
root = "chest_xray"

for split in ["train", "val", "test"]:
    print(f"\n{split} set:")
    for class_name in ["NORMAL", "PNEUMONIA"]:
        path = os.path.join(root, split, class_name)
        num_images = len([
            f for f in os.listdir(path)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        print(f"{class_name}: {num_images}")



train set:
NORMAL: 1341
PNEUMONIA: 3875

val set:
NORMAL: 8
PNEUMONIA: 8

test set:
NORMAL: 234
PNEUMONIA: 390


In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("chest_xray/train", transform=transform)
val_dataset = datasets.ImageFolder("chest_xray/val", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print("训练集大小:", len(train_dataset))
print("验证集大小:", len(val_dataset))

images, labels = next(iter(train_loader))
print("一个 batch 图片 shape:", images.shape)
print("一个 batch 标签:", labels)

训练集大小: 5216
验证集大小: 16
一个 batch 图片 shape: torch.Size([8, 3, 224, 224])
一个 batch 标签: tensor([0, 0, 1, 1, 1, 1, 1, 1])


In [3]:
import os
import random
import shutil
from pathlib import Path

random.seed(42)

src_root = Path("chest_xray/train")
dst_root = Path("chest_xray_split")

val_ratio = 0.2

classes = ["NORMAL", "PNEUMONIA"]

for cls in classes:
    src_dir = src_root / cls

    images = [
        p for p in src_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    random.shuffle(images)

    val_num = int(len(images) * val_ratio)
    val_images = images[:val_num]
    train_images = images[val_num:]

    for split_name, split_images in [
        ("train", train_images),
        ("val", val_images)
    ]:
        dst_dir = dst_root / split_name / cls
        dst_dir.mkdir(parents=True, exist_ok=True)

        for img_path in split_images:
            shutil.copy2(img_path, dst_dir / img_path.name)

    print(cls)
    print("  train:", len(train_images))
    print("  val:", len(val_images))

NORMAL
  train: 1073
  val: 268
PNEUMONIA
  train: 3100
  val: 775


In [4]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("chest_xray_split/train", transform=transform)
val_dataset = datasets.ImageFolder("chest_xray_split/val", transform=transform)

print(train_dataset.classes)
print(train_dataset.class_to_idx)
print(len(train_dataset), len(val_dataset))

loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
images, labels = next(iter(loader))

print(images.shape)
print(labels)

['NORMAL', 'PNEUMONIA']
{'NORMAL': 0, 'PNEUMONIA': 1}
4173 1043
torch.Size([8, 3, 224, 224])
tensor([1, 0, 0, 1, 1, 1, 1, 1])


In [1]:
import os
from pathlib import Path

# 数据集路径
train_root = Path("./chest_xray_split/train")
val_root   = Path("./chest_xray_split/val")
test_root  = Path("./chest_xray/chest_xray/test")  # 官方测试集

datasets = {
    "train": train_root,
    "val": val_root,
    "test": test_root
}

classes = ["NORMAL", "PNEUMONIA"]

def get_file_set(root, cls):
    """返回指定类别下所有文件名集合"""
    cls_path = root / cls
    return set(p.name for p in cls_path.iterdir() if p.is_file())

# 统计每个类别数量
print("=== 数据集数量统计 ===")
for ds_name, root in datasets.items():
    print(f"\n{ds_name.upper()}:")
    for cls in classes:
        file_set = get_file_set(root, cls)
        print(f"  {cls}: {len(file_set)} 张图片")

# 检查重复文件
print("\n=== 数据集重复检查 ===")
for cls in classes:
    train_files = get_file_set(train_root, cls)
    val_files   = get_file_set(val_root, cls)
    test_files  = get_file_set(test_root, cls)

    train_val_overlap = train_files & val_files
    train_test_overlap = train_files & test_files
    val_test_overlap = val_files & test_files

    print(f"\n类别: {cls}")
    print(f"  train ∩ val   = {len(train_val_overlap)}")
    if train_val_overlap:
        print(f"    样例: {list(train_val_overlap)[:10]}")
    print(f"  train ∩ test  = {len(train_test_overlap)}")
    if train_test_overlap:
        print(f"    样例: {list(train_test_overlap)[:10]}")
    print(f"  val ∩ test    = {len(val_test_overlap)}")
    if val_test_overlap:
        print(f"    样例: {list(val_test_overlap)[:10]}")

print("\n检查完成。")

=== 数据集数量统计 ===

TRAIN:
  NORMAL: 1073 张图片
  PNEUMONIA: 3100 张图片

VAL:
  NORMAL: 268 张图片
  PNEUMONIA: 775 张图片

TEST:
  NORMAL: 234 张图片
  PNEUMONIA: 390 张图片

=== 数据集重复检查 ===

类别: NORMAL
  train ∩ val   = 0
  train ∩ test  = 0
  val ∩ test    = 0

类别: PNEUMONIA
  train ∩ val   = 0
  train ∩ test  = 0
  val ∩ test    = 0

检查完成。
